# Service + Repository Pattern
**Author:** Venkatesh Aleti

## Definition
Splitting backend code into **three layers**, each with one job:
- **Route:** handles HTTP and schema validation. Kept thin.
- **Service:** holds the business rules.
- **Repository:** handles database access only, with no rules.

## Real-world example
Placing an e-commerce order: the route receives the request, the service checks the rules (quantity, product exists, enough stock), and the repositories run the SQL. Stock update and order insert happen in one transaction.

## Key points
- Repositories are **injected** into the service, not created inside it.
- Request and response models are **schemas**, used by the route.
- Business errors are **custom exceptions**, which the route maps to status codes.
- You can test the service with a fake repository and swap databases without touching rules.
- In FastAPI, `Depends` does the wiring.

## Interview line
"Routes handle HTTP, services hold business rules, and repositories handle data access. The repository is injected, so I can test with a fake and swap databases easily."

## 1. Schemas and Exceptions
`OrderRequest` is what the client sends, `OrderResponse` is what the API returns. Custom exceptions represent business errors.

In [ ]:
import sqlite3
from dataclasses import dataclass

print("=" * 65)
print("   REAL-WORLD SERVICE + REPOSITORY: E-COMMERCE ORDER FLOW")
print("   (SQLite in-memory = real SQL, no server needed)")
print("=" * 65)

# ================= SCHEMAS (Pydantic models in FastAPI) =================
@dataclass
class OrderRequest:
    user_id: int
    product_id: int
    quantity: int

@dataclass
class OrderResponse:
    order_id: int
    product: str
    quantity: int
    total: float

# ================= CUSTOM EXCEPTIONS (business errors) =================
class NotFoundError(Exception): pass          # -> 404
class ValidationError(Exception): pass        # -> 400
class OutOfStockError(Exception): pass        # -> 409

print("[CELL 1] schemas + exceptions defined")

   REAL-WORLD SERVICE + REPOSITORY: E-COMMERCE ORDER FLOW
   (SQLite in-memory = real SQL, no server needed)
[CELL 1] schemas + exceptions defined


## 2. Database Setup and Repositories
`get_connection()` creates an in-memory SQLite database. One repository per table, SQL only. Both share the **same connection**, so one transaction covers both.

In [ ]:
# ================= DB SETUP (in-memory) =================
def get_connection():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL, stock INTEGER);
        CREATE TABLE orders   (id INTEGER PRIMARY KEY AUTOINCREMENT,
                               user_id INTEGER, product_id INTEGER, quantity INTEGER, total REAL);
        INSERT INTO products VALUES (1, 'Laptop', 50000, 3);
        INSERT INTO products VALUES (2, 'Mouse',    500, 10);
    """)
    print("[DB] in-memory DB created: products(Laptop stock=3, Mouse stock=10), orders(empty)")
    return conn

# ================= REPOSITORIES: SQL only, no rules =================
class ProductRepository:
    def __init__(self, conn):
        self.conn = conn
    def get_by_id(self, product_id):
        print(f"   [PRODUCT-REPO] SELECT * FROM products WHERE id={product_id}")
        return self.conn.execute("SELECT * FROM products WHERE id=?", (product_id,)).fetchone()
    def reduce_stock(self, product_id, qty):
        print(f"   [PRODUCT-REPO] UPDATE products SET stock = stock - {qty} WHERE id={product_id}")
        self.conn.execute("UPDATE products SET stock = stock - ? WHERE id=?", (qty, product_id))

class OrderRepository:
    def __init__(self, conn):
        self.conn = conn
    def create(self, user_id, product_id, qty, total):
        print(f"   [ORDER-REPO] INSERT INTO orders (user={user_id}, product={product_id}, qty={qty}, total={total})")
        cur = self.conn.execute(
            "INSERT INTO orders (user_id, product_id, quantity, total) VALUES (?,?,?,?)",
            (user_id, product_id, qty, total))
        return cur.lastrowid

print("[CELL 2] DB setup + repositories defined")

[CELL 2] DB setup + repositories defined


## 3. Service
Checks three rules, then reduces stock and creates the order in **one transaction**: commit if both succeed, roll back if either fails.

In [ ]:
# ================= SERVICE: business rules + transaction =================
class OrderService:
    def __init__(self, conn, product_repo, order_repo):     # injected
        self.conn = conn
        self.products = product_repo
        self.orders = order_repo
        print("[SERVICE] OrderService created with injected ProductRepo + OrderRepo")

    def place_order(self, req: OrderRequest) -> OrderResponse:
        print(f"   [SERVICE] RULE 1: quantity must be > 0  (got {req.quantity})")
        if req.quantity <= 0:
            raise ValidationError("Quantity must be greater than 0")

        product = self.products.get_by_id(req.product_id)
        print("   [SERVICE] RULE 2: product must exist")
        if product is None:
            raise NotFoundError(f"Product {req.product_id} not found")

        print(f"   [SERVICE] RULE 3: enough stock? have={product['stock']}, want={req.quantity}")
        if product["stock"] < req.quantity:
            raise OutOfStockError(f"Only {product['stock']} {product['name']} left")

        total = product["price"] * req.quantity
        print(f"   [SERVICE] total = {product['price']} x {req.quantity} = {total}")

        print("   [SERVICE] BEGIN TRANSACTION (stock + order must succeed together)")
        try:
            self.products.reduce_stock(req.product_id, req.quantity)
            order_id = self.orders.create(req.user_id, req.product_id, req.quantity, total)
            self.conn.commit()
            print("   [SERVICE] COMMIT -> both changes saved")
        except Exception as e:
            self.conn.rollback()
            print(f"   [SERVICE] ROLLBACK -> nothing saved ({e})")
            raise
        return OrderResponse(order_id, product["name"], req.quantity, total)

print("[CELL 3] OrderService defined")

[CELL 3] OrderService defined


## 4. Route, Wiring and Requests
`post_orders` acts as the route: parse the body, call the service, map exceptions to 400, 404 and 409. The wiring step injects the repositories.

In [ ]:
# ================= ROUTE: HTTP only, maps errors to status codes =================
def post_orders(service, body: dict):
    print(f"\n[ROUTE] POST /orders  body={body}")
    req = OrderRequest(**body)                      # schema parsing
    try:
        resp = service.place_order(req)
        print(f"[ROUTE] 201 Created -> {resp}")
    except ValidationError as e:
        print(f"[ROUTE] 400 Bad Request -> {e}")
    except NotFoundError as e:
        print(f"[ROUTE] 404 Not Found -> {e}")
    except OutOfStockError as e:
        print(f"[ROUTE] 409 Conflict -> {e}")

def show_tables(conn):
    print("\n[DB-STATE] products:", [dict(r) for r in conn.execute("SELECT * FROM products")])
    print("[DB-STATE] orders:  ", [dict(r) for r in conn.execute("SELECT * FROM orders")])

# ================= WIRING (FastAPI does this with Depends) =================
conn = get_connection()
service = OrderService(conn, ProductRepository(conn), OrderRepository(conn))

print("\n" + "#" * 65)
print("# REQUESTS")
print("#" * 65)
post_orders(service, {"user_id": 7, "product_id": 1, "quantity": 2})   # success
post_orders(service, {"user_id": 8, "product_id": 1, "quantity": 5})   # out of stock
post_orders(service, {"user_id": 9, "product_id": 99, "quantity": 1})  # not found
post_orders(service, {"user_id": 9, "product_id": 2, "quantity": 0})   # invalid qty
post_orders(service, {"user_id": 9, "product_id": 2, "quantity": 4})   # success

show_tables(conn)

print("\n" + "=" * 65)
print("KEY TAKEAWAY")
print("=" * 65)
print("Route   -> parse schema, call service, map errors to 400/404/409")
print("Service -> rules (qty, exists, stock) + transaction commit/rollback")
print("Repos   -> one per table, SQL only")
print("Service raises business exceptions; route turns them into HTTP codes")

[DB] in-memory DB created: products(Laptop stock=3, Mouse stock=10), orders(empty)
[SERVICE] OrderService created with injected ProductRepo + OrderRepo

#################################################################
# REQUESTS
#################################################################

[ROUTE] POST /orders  body={'user_id': 7, 'product_id': 1, 'quantity': 2}
   [SERVICE] RULE 1: quantity must be > 0  (got 2)
   [PRODUCT-REPO] SELECT * FROM products WHERE id=1
   [SERVICE] RULE 2: product must exist
   [SERVICE] RULE 3: enough stock? have=3, want=2
   [SERVICE] total = 50000.0 x 2 = 100000.0
   [SERVICE] BEGIN TRANSACTION (stock + order must succeed together)
   [PRODUCT-REPO] UPDATE products SET stock = stock - 2 WHERE id=1
   [ORDER-REPO] INSERT INTO orders (user=7, product=1, qty=2, total=100000.0)
   [SERVICE] COMMIT -> both changes saved
[ROUTE] 201 Created -> OrderResponse(order_id=1, product='Laptop', quantity=2, total=100000.0)

[ROUTE] POST /orders  body={'user_id': 8

## 5. Query the Database
Run your own SQL on the same connection, including a JOIN between orders and products.

In [ ]:
# ================= CELL 5: run your own SQL queries =================
def run_query(sql):
    print(f"\n[QUERY] {sql}")
    rows = conn.execute(sql).fetchall()
    if not rows:
        print("   (no rows)")
        return
    cols = rows[0].keys()
    print("   " + " | ".join(cols))
    print("   " + "-" * 50)
    for r in rows:
        print("   " + " | ".join(str(r[c]) for c in cols))
    print(f"   [{len(rows)} row(s)]")

run_query("SELECT * FROM products")
run_query("SELECT * FROM orders")

# a JOIN: which product did each order buy?
run_query("""
    SELECT o.id AS order_id, o.user_id, p.name AS product, o.quantity, o.total
    FROM orders o
    JOIN products p ON p.id = o.product_id
""")


[QUERY] SELECT * FROM products
   id | name | price | stock
   --------------------------------------------------
   1 | Laptop | 50000.0 | 1
   2 | Mouse | 500.0 | 6
   [2 row(s)]

[QUERY] SELECT * FROM orders
   id | user_id | product_id | quantity | total
   --------------------------------------------------
   1 | 7 | 1 | 2 | 100000.0
   2 | 9 | 2 | 4 | 2000.0
   [2 row(s)]

[QUERY] 
    SELECT o.id AS order_id, o.user_id, p.name AS product, o.quantity, o.total
    FROM orders o
    JOIN products p ON p.id = o.product_id

   order_id | user_id | product | quantity | total
   --------------------------------------------------
   1 | 7 | Laptop | 2 | 100000.0
   2 | 9 | Mouse | 4 | 2000.0
   [2 row(s)]
